In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/Social_media_impact_on_life.csv")
df.head()

,Student_ID,Age,Gender,Academic_Level,Primary_Platform,Daily_Usage_Hours,Weekend_Extra_Hours,Device_Type,Sleep_Duration_Hours,Sleep_Quality_Score,Late_Night_Usage,Social_Comparison_Frequency,Perceived_Stress_Score,Mental_Health_Index,Academic_Performance_GPA,Overall_Impact
0,STU_2024000,19,Female,Undergraduate,Snapchat,7.5,1.5,Smartphone,5.9,2,True,Always,20.0,73,2.73,Neutral
1,STU_2024001,19,Female,Undergraduate,Instagram,4.6,1.3,Smartphone,7.9,3,True,Sometimes,17.0,90,3.21,Beneficial
2,STU_2024002,17,Female,High School,TikTok,10.9,1.7,Smartphone,5.2,1,True,Frequently,18.0,70,3.00,Neutral
3,STU_2024003,16,Female,High School,YouTube,6.0,2.3,Tablet,7.6,4,True,Sometimes,15.0,82,3.42,Beneficial
4,STU_2024004,17,Male,High School,LinkedIn,3.3,2.6,Smartphone,7.6,4,True,Never,16.0,84,3.41,Beneficial


In [3]:
X = df.drop(columns=["Student_ID","Age","Gender","Academic_Level","Mental_Health_Index"])
y = df["Mental_Health_Index"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((3600, 11), (900, 11), (3600,), (900,))

In [4]:
preprocessor = ColumnTransformer([
    ('numeric', num_pipe(), [
        "Daily_Usage_Hours",
        "Weekend_Extra_Hours",
        "Sleep_Duration_Hours",
        "Sleep_Quality_Score",
        # "Social_Comparison_Frequency",  <-- Removed from here
        "Perceived_Stress_Score",
        "Academic_Performance_GPA"
    ]),
    ('categoric', cat_pipe(encoder='onehot'), [
        "Primary_Platform",
        "Device_Type",
        "Late_Night_Usage",
        "Overall_Impact",
        "Social_Comparison_Frequency"    # <-- Moved here!
    ]),
])

In [5]:
from sklearn.neighbors import KNeighborsRegressor
pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', KNeighborsRegressor())
])

In [6]:
from sklearn.model_selection import GridSearchCV
from jcopml.tuning import grid_search_params as gsp

gsp = {
    'algo__n_neighbors': [3, 5, 7, 9, 11],
    'algo__weights': ['uniform', 'distance'],
    'algo__p': [1, 2]
}

model = GridSearchCV(pipeline, gsp, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1)
model.fit(X_train, y_train)

print(model.best_params_)
print(model.score(X_train, y_train), model.best_score_, model.score(X_test, y_test))
print(y.describe())

Fitting 5 folds for each of 20 candidates, totalling 100 fits
{'algo__n_neighbors': 11, 'algo__p': 2, 'algo__weights': 'uniform'}
-4.396388888888889 -4.859141414141415 -4.942828282828282
count    4500.000000
mean       79.873111
std        12.876044
min        32.000000
25%        72.000000
50%        82.000000
75%        89.000000
max        98.000000
Name: Mental_Health_Index, dtype: float64
